# 02 — Preprocessing

Spatial preprocessing of the hourly wind data for the Persian Gulf and Gulf of Oman.

This notebook uses the final study grid created in `01_data_audit.ipynb` and extracts the official 512 marine grid cells from the annual NetCDF files.

Main tasks:
- Load the final study grid
- Validate raw NetCDF files
- Extract the 512 study-area cells
- Keep hourly u10 and v10 data
- Preserve spatial and temporal information
- Save processed yearly NetCDF files

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

PROJECT_ROOT = Path("..")

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
YEARLY_DIR = PROCESSED_DIR / "yearly"

STUDY_GRID_PATH = PROCESSED_DIR / "study_grid.csv"

YEARLY_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Project root:", PROJECT_ROOT.resolve())
print("Raw data:", RAW_DIR.resolve())
print("Study grid:", STUDY_GRID_PATH.resolve())
print("Output:", YEARLY_DIR.resolve())

Project root: /home/mike/Desktop/wind_paper
Raw data: /home/mike/Desktop/wind_paper/data/raw
Study grid: /home/mike/Desktop/wind_paper/data/processed/study_grid.csv
Output: /home/mike/Desktop/wind_paper/data/processed/yearly


In [2]:
study_grid = pd.read_csv(STUDY_GRID_PATH)

print("Shape:", study_grid.shape)
print("\nColumns:")
print(study_grid.columns.tolist())

print("\nFirst 5 rows:")
display(study_grid.head())

print("\nBasin counts:")
print(study_grid["basin"].value_counts())

print("\nDuplicate study_id:", study_grid["study_id"].duplicated().sum())
print("Duplicate grid_id:", study_grid["grid_id"].duplicated().sum())
print(
    "Duplicate lat/lon:",
    study_grid.duplicated(["latitude", "longitude"]).sum()
)

Shape: (512, 5)

Columns:
['study_id', 'grid_id', 'latitude', 'longitude', 'basin']

First 5 rows:


,study_id,grid_id,latitude,longitude,basin
0,0,594,22.75,59.50,Gulf of Oman
1,1,595,22.75,59.75,Gulf of Oman
2,2,578,23.00,59.25,Gulf of Oman
3,3,579,23.00,59.50,Gulf of Oman
4,4,580,23.00,59.75,Gulf of Oman



Basin counts:
basin
Persian Gulf    357
Gulf of Oman    155
Name: count, dtype: int64

Duplicate study_id: 0
Duplicate grid_id: 0
Duplicate lat/lon: 0


## Validate the study grid

In this step, the final study grid is validated before processing the raw wind data.

The validation checks:

- Required columns exist
- The study grid contains exactly 512 cells
- `study_id` values are unique
- `grid_id` values are unique
- There are no duplicate latitude/longitude pairs
- The Persian Gulf contains 357 cells
- The Gulf of Oman contains 155 cells

This validation ensures that the spatial domain used in the preprocessing stage is consistent with the final grid created during the data audit.

In [3]:
# Validate the official study grid

required_columns = {
    "study_id",
    "grid_id",
    "latitude",
    "longitude",
    "basin",
}

missing_columns = required_columns.difference(study_grid.columns)

assert not missing_columns, (
    f"Missing columns: {sorted(missing_columns)}"
)

assert len(study_grid) == 512, (
    f"Expected 512 study cells, found {len(study_grid)}"
)

assert study_grid["study_id"].nunique() == 512

assert study_grid["grid_id"].nunique() == 512

assert (
    study_grid[["latitude", "longitude"]]
    .drop_duplicates()
    .shape[0]
    == 512
)

basin_counts = study_grid["basin"].value_counts()

assert basin_counts["Persian Gulf"] == 357
assert basin_counts["Gulf of Oman"] == 155

study_grid = (
    study_grid
    .sort_values("study_id")
    .reset_index(drop=True)
)

print("Study grid validation: PASSED")
print("Total study cells:", len(study_grid))
print("Persian Gulf:", basin_counts["Persian Gulf"])
print("Gulf of Oman:", basin_counts["Gulf of Oman"])

Study grid validation: PASSED
Total study cells: 512
Persian Gulf: 357
Gulf of Oman: 155


## Discover the raw NetCDF files

Before starting the preprocessing procedure, the annual raw NetCDF files are identified and inspected.

The project is expected to contain one NetCDF file for each year from 2005 to 2020.

In this step, we:

- Search the raw-data directory for `.nc` files
- Count the available files
- Display their filenames
- Confirm that all 16 annual datasets are available before continuing

In [4]:
# Discover all raw NetCDF files

nc_files = sorted(RAW_DIR.glob("*.nc"))

print("Number of NetCDF files:", len(nc_files))

print("\nAvailable files:")
for file in nc_files:
    print(file.name)

assert len(nc_files) == 16, (
    f"Expected 16 annual NetCDF files, found {len(nc_files)}"
)

print("\nRaw NetCDF file check: PASSED")

Number of NetCDF files: 16

Available files:
2005-01-01_to_2005-12-31.nc
2006-01-01_to_2006-12-31.nc
2007-01-01_to_2007-12-31.nc
2008-01-01_to_2008-12-31.nc
2009-01-01_to_2009-12-31.nc
2010-01-01_to_2010-12-31.nc
2011-01-01_to_2011-12-31.nc
2012-01-01_to_2012-12-31.nc
2013-01-01_to_2013-12-31.nc
2014-01-01_to_2014-12-31.nc
2015-01-01_to_2015-12-31.nc
2016-01-01_to_2016-12-31.nc
2017-01-01_to_2017-12-31.nc
2018-01-01_to_2018-12-31.nc
2019-01-01_to_2019-12-31.nc
2020-01-01_to_2020-12-31.nc

Raw NetCDF file check: PASSED


## Validate the annual time coverage

The raw dataset should contain one NetCDF file for every year from 2005 through 2020.

In this step, the year is extracted from each filename and compared with the expected sequence.

This check ensures that:

- All years from 2005 to 2020 are available
- No year is missing
- No unexpected year is present
- The annual files are correctly ordered

In [5]:
# Validate annual file coverage

expected_years = list(range(2005, 2021))

found_years = [
    int(file.name.split("-")[0])
    for file in nc_files
]

print("Expected years:")
print(expected_years)

print("\nFound years:")
print(found_years)

missing_years = sorted(
    set(expected_years) - set(found_years)
)

unexpected_years = sorted(
    set(found_years) - set(expected_years)
)

print("\nMissing years:", missing_years)
print("Unexpected years:", unexpected_years)

assert found_years == expected_years, (
    "Annual NetCDF files do not match the expected 2005–2020 sequence."
)

print("\nAnnual coverage check: PASSED")

Expected years:
[2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020]

Found years:
[2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020]

Missing years: []
Unexpected years: []

Annual coverage check: PASSED


## Inspect a sample raw NetCDF file

Before processing all annual datasets, one raw NetCDF file is opened and inspected in detail.

The 2005 dataset is used as a representative sample.

In this step, we examine:

- Dataset dimensions
- Available data variables
- Coordinate variables
- Time coverage
- Latitude and longitude ranges
- The structure of the `u10` and `v10` wind components

This inspection confirms that the raw dataset has the expected structure before applying the preprocessing procedure to all years.

In [6]:
# Inspect the first raw NetCDF file

sample_file = nc_files[0]

print("Sample file:")
print(sample_file.name)

with xr.open_dataset(sample_file) as ds_sample:

    print("\nDataset structure:")
    print(ds_sample)

    print("\nDimensions:")
    print(dict(ds_sample.sizes))

    print("\nData variables:")
    print(list(ds_sample.data_vars))

    print("\nCoordinates:")
    print(list(ds_sample.coords))

    print("\nTime range:")
    print("Start:", ds_sample["valid_time"].values[0])
    print("End  :", ds_sample["valid_time"].values[-1])

    print("\nLatitude range:")
    print(
        float(ds_sample["latitude"].min()),
        "to",
        float(ds_sample["latitude"].max())
    )

    print("\nLongitude range:")
    print(
        float(ds_sample["longitude"].min()),
        "to",
        float(ds_sample["longitude"].max())
    )

    print("\nu10 dimensions:")
    print(ds_sample["u10"].dims)

    print("\nv10 dimensions:")
    print(ds_sample["v10"].dims)

Sample file:
2005-01-01_to_2005-12-31.nc

Dataset structure:
<xarray.Dataset> Size: 169MB
Dimensions:     (valid_time: 8760, latitude: 37, longitude: 65)
Coordinates:
    number      int64 8B ...
  * valid_time  (valid_time) datetime64[ns] 70kB 2005-01-01 ... 2005-12-31T23...
  * latitude    (latitude) float64 296B 31.0 30.75 30.5 ... 22.5 22.25 22.0
  * longitude   (longitude) float64 520B 47.0 47.25 47.5 ... 62.5 62.75 63.0
    expver      (valid_time) <U4 140kB ...
Data variables:
    u10         (valid_time, latitude, longitude) float32 84MB ...
    v10         (valid_time, latitude, longitude) float32 84MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2025-10-18T05:14 GRIB to CDM+CF via cfgrib-0.9.1...

Dimensions:
{'valid_ti

## Validate the sample dataset

After inspecting the structure of the sample NetCDF file, several consistency checks are performed before applying the preprocessing workflow to all annual datasets.

This step verifies:

- The spatial grid has a resolution of 0.25 degrees
- The dataset contains hourly observations without time gaps
- The 2005 dataset contains exactly 8,760 hourly records
- The first and last timestamps cover the complete calendar year
- The `u10` and `v10` variables contain no missing values

These checks help ensure that the raw data are temporally and spatially consistent before spatial extraction.

In [7]:
# Validate spatial and temporal consistency of the sample dataset

sample_year = 2005

with xr.open_dataset(sample_file) as ds_sample:

    latitudes = ds_sample["latitude"].values
    longitudes = ds_sample["longitude"].values

    times = pd.DatetimeIndex(
        pd.to_datetime(ds_sample["valid_time"].values)
    )

    # Calculate spatial resolution
    lat_resolution = np.median(
        np.abs(np.diff(latitudes))
    )

    lon_resolution = np.median(
        np.abs(np.diff(longitudes))
    )

    # Calculate temporal differences
    time_differences = pd.Series(times).diff().dropna()

    # Count missing values
    u10_missing = int(
        ds_sample["u10"].isnull().sum().values
    )

    v10_missing = int(
        ds_sample["v10"].isnull().sum().values
    )

print("Latitude resolution :", lat_resolution)
print("Longitude resolution:", lon_resolution)

print("\nNumber of timestamps:", len(times))
print("First timestamp:", times[0])
print("Last timestamp :", times[-1])

print(
    "\nUnique time intervals:",
    time_differences.unique()
)

print("\nu10 missing values:", u10_missing)
print("v10 missing values:", v10_missing)

# Validation checks

assert np.isclose(lat_resolution, 0.25)
assert np.isclose(lon_resolution, 0.25)

assert len(times) == 8760

assert times[0] == pd.Timestamp(
    "2005-01-01 00:00:00"
)

assert times[-1] == pd.Timestamp(
    "2005-12-31 23:00:00"
)

assert (
    time_differences
    == pd.Timedelta(hours=1)
).all()

assert u10_missing == 0
assert v10_missing == 0

print("\nSample dataset validation: PASSED")

Latitude resolution : 0.25
Longitude resolution: 0.25

Number of timestamps: 8760
First timestamp: 2005-01-01 00:00:00
Last timestamp : 2005-12-31 23:00:00

Unique time intervals: <TimedeltaArray>
['0 days 01:00:00']
Length: 1, dtype: timedelta64[ns]

u10 missing values: 0
v10 missing values: 0

Sample dataset validation: PASSED


## Validate all annual raw datasets

After confirming that the 2005 sample file has the expected structure, the same validation procedure is applied to every annual NetCDF file from 2005 to 2020.

For each year, this step verifies:

- The presence of `u10` and `v10`
- The presence of `valid_time`, `latitude`, and `longitude`
- A spatial grid size of 37 × 65
- A spatial resolution of 0.25 degrees
- Complete hourly temporal coverage
- 8,760 hours for normal years and 8,784 hours for leap years
- Correct first and last timestamps for each calendar year
- No duplicate timestamps
- No missing values in `u10` or `v10`

A summary table is generated to confirm that all raw annual datasets are suitable for preprocessing.

In [8]:
# Validate all annual raw NetCDF datasets

def expected_hours_in_year(year):
    """Return the expected number of hourly records for a calendar year."""
    
    start = pd.Timestamp(f"{year}-01-01 00:00:00")
    end = pd.Timestamp(f"{year + 1}-01-01 00:00:00")
    
    return int((end - start) / pd.Timedelta(hours=1))


validation_results = []

for file in nc_files:
    
    year = int(file.name.split("-")[0])
    
    print(f"Checking {year} ...")
    
    with xr.open_dataset(file) as ds:
        
        # Check required variables and coordinates
        assert "u10" in ds.data_vars
        assert "v10" in ds.data_vars
        
        assert "valid_time" in ds.coords
        assert "latitude" in ds.coords
        assert "longitude" in ds.coords
        
        # Check spatial dimensions
        assert ds.sizes["latitude"] == 37
        assert ds.sizes["longitude"] == 65
        
        latitudes = ds["latitude"].values
        longitudes = ds["longitude"].values
        
        lat_resolution = np.median(
            np.abs(np.diff(latitudes))
        )
        
        lon_resolution = np.median(
            np.abs(np.diff(longitudes))
        )
        
        assert np.isclose(lat_resolution, 0.25)
        assert np.isclose(lon_resolution, 0.25)
        
        # Check temporal structure
        times = pd.DatetimeIndex(
            pd.to_datetime(ds["valid_time"].values)
        )
        
        expected_hours = expected_hours_in_year(year)
        
        assert len(times) == expected_hours
        
        assert times[0] == pd.Timestamp(
            f"{year}-01-01 00:00:00"
        )
        
        assert times[-1] == pd.Timestamp(
            f"{year}-12-31 23:00:00"
        )
        
        assert not times.has_duplicates
        
        time_differences = (
            pd.Series(times)
            .diff()
            .dropna()
        )
        
        assert (
            time_differences == pd.Timedelta(hours=1)
        ).all()
        
        # Check missing values
        u10_missing = int(
            ds["u10"].isnull().sum().values
        )
        
        v10_missing = int(
            ds["v10"].isnull().sum().values
        )
        
        assert u10_missing == 0
        assert v10_missing == 0
        
        validation_results.append({
            "year": year,
            "hours": len(times),
            "latitude_points": ds.sizes["latitude"],
            "longitude_points": ds.sizes["longitude"],
            "u10_missing": u10_missing,
            "v10_missing": v10_missing,
            "status": "PASSED"
        })


validation_summary = pd.DataFrame(validation_results)

print("\nValidation summary:")
display(validation_summary)

print(
    "\nAll annual raw datasets validation: PASSED"
)

Checking 2005 ...
Checking 2006 ...
Checking 2007 ...
Checking 2008 ...
Checking 2009 ...
Checking 2010 ...
Checking 2011 ...
Checking 2012 ...
Checking 2013 ...
Checking 2014 ...
Checking 2015 ...
Checking 2016 ...
Checking 2017 ...
Checking 2018 ...
Checking 2019 ...
Checking 2020 ...

Validation summary:


,year,hours,latitude_points,longitude_points,u10_missing,v10_missing,status
0,2005,8760,37,65,0,0,PASSED
1,2006,8760,37,65,0,0,PASSED
2,2007,8760,37,65,0,0,PASSED
3,2008,8784,37,65,0,0,PASSED
4,2009,8760,37,65,0,0,PASSED
5,2010,8760,37,65,0,0,PASSED
6,2011,8760,37,65,0,0,PASSED
7,2012,8784,37,65,0,0,PASSED
8,2013,8760,37,65,0,0,PASSED
9,2014,8760,37,65,0,0,PASSED



All annual raw datasets validation: PASSED


## Extract the official study-area grid

The raw NetCDF files cover the complete rectangular domain from 22°N to 31°N and 47°E to 63°E. However, the final study area contains only the 512 marine grid cells identified during the data-audit stage.

In this step, the raw wind fields are spatially filtered using the official `study_grid.csv`.

The extraction procedure:

- Selects the exact latitude/longitude pairs defined in the study grid
- Preserves the original hourly `u10` and `v10` values
- Maintains the official `study_id` and `grid_id`
- Removes unnecessary auxiliary coordinates such as `number` and `expver`
- Produces a compact dataset with dimensions `valid_time × study_id`

The extraction is first tested on the 2005 dataset before processing all years.

In [9]:
# Extract the official 512 study-grid cells

def extract_study_grid(ds, study_grid_df):
    """
    Extract the exact 512 official study-grid points
    from a raw annual NetCDF dataset.
    """

    # Create paired latitude/longitude indexers
    lat_indexer = xr.DataArray(
        study_grid_df["latitude"].to_numpy(),
        dims="study_id"
    )

    lon_indexer = xr.DataArray(
        study_grid_df["longitude"].to_numpy(),
        dims="study_id"
    )

    # Select only u10 and v10 at the paired study-grid locations
    processed = ds[["u10", "v10"]].sel(
        latitude=lat_indexer,
        longitude=lon_indexer
    )

    # Remove auxiliary coordinates that are not needed
    auxiliary_coords = [
        coord
        for coord in ["number", "expver"]
        if coord in processed.coords
    ]

    if auxiliary_coords:
        processed = processed.drop_vars(auxiliary_coords)

    # Attach official identifiers
    processed = processed.assign_coords(
        study_id=(
            "study_id",
            study_grid_df["study_id"].to_numpy()
        ),
        grid_id=(
            "study_id",
            study_grid_df["grid_id"].to_numpy()
        )
    )

    # Keep a consistent dimension order
    processed = processed.transpose(
        "valid_time",
        "study_id"
    )

    return processed

## Test the spatial extraction on the 2005 dataset

Before applying the extraction procedure to all 16 annual files, the function is tested on the 2005 dataset.

The expected result is:

- 8,760 hourly timestamps
- Exactly 512 study cells
- Two wind variables: `u10` and `v10`
- Latitude and longitude stored for every study cell
- No change to the original wind values

In [10]:
# Test extraction on the 2005 dataset

with xr.open_dataset(sample_file) as ds_sample:

    sample_processed = extract_study_grid(
        ds_sample,
        study_grid
    )

    # Load the small extracted subset into memory
    sample_processed = sample_processed.load()


print("Processed dataset:")
print(sample_processed)

print("\nDimensions:")
print(dict(sample_processed.sizes))

print("\nData variables:")
print(list(sample_processed.data_vars))

print("\nCoordinates:")
print(list(sample_processed.coords))

print("\nNumber of study cells:")
print(sample_processed.sizes["study_id"])

print("\nFirst 5 study IDs:")
print(sample_processed["study_id"].values[:5])

print("\nFirst 5 grid IDs:")
print(sample_processed["grid_id"].values[:5])

print("\nFirst 5 latitude values:")
print(sample_processed["latitude"].values[:5])

print("\nFirst 5 longitude values:")
print(sample_processed["longitude"].values[:5])

Processed dataset:
<xarray.Dataset> Size: 36MB
Dimensions:     (valid_time: 8760, study_id: 512)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 70kB 2005-01-01 ... 2005-12-31T23...
    latitude    (study_id) float64 4kB 22.75 22.75 23.0 23.0 ... 30.0 30.0 30.25
    longitude   (study_id) float64 4kB 59.5 59.75 59.25 59.5 ... 49.75 50.0 49.0
  * study_id    (study_id) int64 4kB 0 1 2 3 4 5 6 ... 506 507 508 509 510 511
    grid_id     (study_id) int64 4kB 594 595 578 579 580 581 561 ... 5 6 7 8 9 1
Data variables:
    u10         (valid_time, study_id) float32 18MB 3.677 5.39 ... -1.27 1.825
    v10         (valid_time, study_id) float32 18MB -5.22 -6.883 ... -3.389
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2025-10-18T05:

## Verify spatial alignment with the official study grid

After extracting the 512 study-area cells, the spatial alignment must be verified.

This step confirms that:

- `study_id` values are in the exact expected order
- `grid_id` values match the official study grid
- Latitude values match the original `study_grid.csv`
- Longitude values match the original `study_grid.csv`
- No spatial point has been reordered or shifted during extraction

This validation is important because any spatial misalignment could propagate incorrect location information into later feature-engineering and modeling stages.

In [11]:
# Verify exact spatial alignment

expected_study_ids = study_grid["study_id"].to_numpy()
expected_grid_ids = study_grid["grid_id"].to_numpy()
expected_latitudes = study_grid["latitude"].to_numpy()
expected_longitudes = study_grid["longitude"].to_numpy()

actual_study_ids = sample_processed["study_id"].values
actual_grid_ids = sample_processed["grid_id"].values
actual_latitudes = sample_processed["latitude"].values
actual_longitudes = sample_processed["longitude"].values

study_id_match = np.array_equal(
    actual_study_ids,
    expected_study_ids
)

grid_id_match = np.array_equal(
    actual_grid_ids,
    expected_grid_ids
)

latitude_match = np.allclose(
    actual_latitudes,
    expected_latitudes
)

longitude_match = np.allclose(
    actual_longitudes,
    expected_longitudes
)

print("study_id match :", study_id_match)
print("grid_id match  :", grid_id_match)
print("latitude match :", latitude_match)
print("longitude match:", longitude_match)

assert study_id_match
assert grid_id_match
assert latitude_match
assert longitude_match

print("\nSpatial alignment validation: PASSED")

study_id match : True
grid_id match  : True
latitude match : True
longitude match: True

Spatial alignment validation: PASSED


## Verify that wind values are preserved after spatial extraction

The spatial extraction should only reduce the domain from the full rectangular grid to the 512 official study cells.

It must not modify the original wind values.

In this step, several study-grid points are selected and their `u10` and `v10` values are compared directly with the corresponding values in the raw 2005 NetCDF file.

This check confirms that:

- Spatial selection is exact
- No interpolation has been applied
- `u10` values are preserved
- `v10` values are preserved
- The extracted dataset contains the original hourly wind measurements at the selected study locations

In [12]:
# Verify that u10 and v10 values are unchanged after extraction

test_study_ids = [0, 100, 250, 511]

with xr.open_dataset(sample_file) as ds_raw:

    for sid in test_study_ids:

        lat = float(
            study_grid.loc[
                study_grid["study_id"] == sid,
                "latitude"
            ].iloc[0]
        )

        lon = float(
            study_grid.loc[
                study_grid["study_id"] == sid,
                "longitude"
            ].iloc[0]
        )

        raw_u10 = (
            ds_raw["u10"]
            .sel(
                latitude=lat,
                longitude=lon
            )
            .values
        )

        raw_v10 = (
            ds_raw["v10"]
            .sel(
                latitude=lat,
                longitude=lon
            )
            .values
        )

        processed_u10 = (
            sample_processed["u10"]
            .sel(study_id=sid)
            .values
        )

        processed_v10 = (
            sample_processed["v10"]
            .sel(study_id=sid)
            .values
        )

        u10_match = np.allclose(
            raw_u10,
            processed_u10,
            equal_nan=True
        )

        v10_match = np.allclose(
            raw_v10,
            processed_v10,
            equal_nan=True
        )

        print(
            f"study_id={sid:3d} | "
            f"lat={lat:5.2f} | "
            f"lon={lon:5.2f} | "
            f"u10={u10_match} | "
            f"v10={v10_match}"
        )

        assert u10_match
        assert v10_match


print(
    "\nWind-value preservation validation: PASSED"
)

study_id=  0 | lat=22.75 | lon=59.50 | u10=True | v10=True
study_id=100 | lat=24.75 | lon=61.25 | u10=True | v10=True
study_id=250 | lat=25.75 | lon=53.25 | u10=True | v10=True
study_id=511 | lat=30.25 | lon=49.00 | u10=True | v10=True

Wind-value preservation validation: PASSED


## Process and save all annual datasets

After validating the spatial extraction procedure, preprocessing is applied to all annual NetCDF files from 2005 to 2020.

For each year, the workflow:

- Opens the original annual NetCDF file
- Extracts the exact 512 official study-grid cells
- Preserves the hourly `u10` and `v10` values
- Preserves `study_id`, `grid_id`, latitude, longitude, and time
- Removes unnecessary auxiliary coordinates
- Validates the extracted dataset
- Saves the processed dataset as a separate annual NetCDF file

The processed files are stored in:

`data/processed/yearly/`

No feature engineering, scaling, lag construction, or target generation is performed at this stage.

In [13]:
# Process and save all annual datasets

processing_results = []

for file in nc_files:

    year = int(file.name.split("-")[0])

    output_file = YEARLY_DIR / f"{year}.nc"

    print(f"Processing {year} ...")

    with xr.open_dataset(file) as ds_raw:

        # Extract the official 512 study cells
        processed = extract_study_grid(
            ds_raw,
            study_grid
        )

        # Load extracted data before closing the raw file
        processed = processed.load()

    # -----------------------------
    # Validate processed structure
    # -----------------------------

    expected_hours = expected_hours_in_year(year)

    assert processed.sizes["valid_time"] == expected_hours

    assert processed.sizes["study_id"] == 512

    assert np.array_equal(
        processed["study_id"].values,
        study_grid["study_id"].to_numpy()
    )

    assert np.array_equal(
        processed["grid_id"].values,
        study_grid["grid_id"].to_numpy()
    )

    assert np.allclose(
        processed["latitude"].values,
        study_grid["latitude"].to_numpy()
    )

    assert np.allclose(
        processed["longitude"].values,
        study_grid["longitude"].to_numpy()
    )

    u10_missing = int(
        processed["u10"]
        .isnull()
        .sum()
        .values
    )

    v10_missing = int(
        processed["v10"]
        .isnull()
        .sum()
        .values
    )

    assert u10_missing == 0
    assert v10_missing == 0

    # -----------------------------
    # Add useful metadata
    # -----------------------------

    processed.attrs = {
        "title": (
            "Hourly 10-m wind components over the "
            "Persian Gulf and Gulf of Oman study grid"
        ),
        "year": year,
        "study_cells": 512,
        "spatial_resolution": "0.25 degrees",
        "temporal_resolution": "1 hour",
        "preprocessing": (
            "Spatial extraction to the official "
            "512-cell study grid"
        )
    }

    processed["u10"].attrs["units"] = "m s-1"
    processed["v10"].attrs["units"] = "m s-1"

    # -----------------------------
    # Save annual NetCDF
    # -----------------------------

    processed.to_netcdf(
        output_file
    )

    processing_results.append({
        "year": year,
        "hours": processed.sizes["valid_time"],
        "study_cells": processed.sizes["study_id"],
        "u10_missing": u10_missing,
        "v10_missing": v10_missing,
        "output_file": output_file.name,
        "status": "SAVED"
    })

    print(
        f"Saved: {output_file.name}"
    )


processing_summary = pd.DataFrame(
    processing_results
)

print("\nProcessing summary:")
display(processing_summary)

print(
    "\nAll annual datasets processed successfully."
)

Processing 2005 ...
Saved: 2005.nc
Processing 2006 ...
Saved: 2006.nc
Processing 2007 ...
Saved: 2007.nc
Processing 2008 ...
Saved: 2008.nc
Processing 2009 ...
Saved: 2009.nc
Processing 2010 ...
Saved: 2010.nc
Processing 2011 ...
Saved: 2011.nc
Processing 2012 ...
Saved: 2012.nc
Processing 2013 ...
Saved: 2013.nc
Processing 2014 ...
Saved: 2014.nc
Processing 2015 ...
Saved: 2015.nc
Processing 2016 ...
Saved: 2016.nc
Processing 2017 ...
Saved: 2017.nc
Processing 2018 ...
Saved: 2018.nc
Processing 2019 ...
Saved: 2019.nc
Processing 2020 ...
Saved: 2020.nc

Processing summary:


,year,hours,study_cells,u10_missing,v10_missing,output_file,status
0,2005,8760,512,0,0,2005.nc,SAVED
1,2006,8760,512,0,0,2006.nc,SAVED
2,2007,8760,512,0,0,2007.nc,SAVED
3,2008,8784,512,0,0,2008.nc,SAVED
4,2009,8760,512,0,0,2009.nc,SAVED
5,2010,8760,512,0,0,2010.nc,SAVED
6,2011,8760,512,0,0,2011.nc,SAVED
7,2012,8784,512,0,0,2012.nc,SAVED
8,2013,8760,512,0,0,2013.nc,SAVED
9,2014,8760,512,0,0,2014.nc,SAVED



All annual datasets processed successfully.


## Save the preprocessing summary

A preprocessing summary is saved as a CSV file for reproducibility and project documentation.

The summary records, for each year:

- Number of hourly observations
- Number of study-grid cells
- Missing-value counts for `u10` and `v10`
- Name of the processed NetCDF file
- Processing status

The summary file is stored under `outputs/tables/`.

In [14]:
# Save preprocessing summary

TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"

TABLES_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PREPROCESSING_SUMMARY_PATH = (
    TABLES_DIR / "preprocessing_summary.csv"
)

processing_summary.to_csv(
    PREPROCESSING_SUMMARY_PATH,
    index=False
)

print(
    "Saved preprocessing summary:"
)

print(
    PREPROCESSING_SUMMARY_PATH.resolve()
)

print(
    "\nNumber of records:",
    len(processing_summary)
)

Saved preprocessing summary:
/home/mike/Desktop/wind_paper/outputs/tables/preprocessing_summary.csv

Number of records: 16


## Final integrity check of processed files

After saving the processed annual NetCDF datasets, all output files are reopened from disk and validated independently.

This final check verifies that:

- Exactly 16 processed annual files exist
- The files cover the complete period from 2005 to 2020
- Each file contains exactly 512 study-grid cells
- Normal years contain 8,760 hourly records
- Leap years contain 8,784 hourly records
- `u10` and `v10` contain no missing values
- `study_id`, `grid_id`, latitude, and longitude remain aligned with the official study grid
- The first and last timestamps correspond to the complete calendar year

This step confirms that the preprocessing outputs were written correctly and are ready for the next stage of the project.

In [15]:
# Final integrity check of all saved processed NetCDF files

processed_files = sorted(
    YEARLY_DIR.glob("*.nc")
)

print(
    "Number of processed files:",
    len(processed_files)
)

assert len(processed_files) == 16, (
    f"Expected 16 processed files, "
    f"found {len(processed_files)}"
)

processed_years = [
    int(file.stem)
    for file in processed_files
]

assert processed_years == list(
    range(2005, 2021)
)

final_check_results = []

for file in processed_files:

    year = int(file.stem)

    print(f"Checking {year} ...")

    with xr.open_dataset(file) as ds:

        expected_hours = expected_hours_in_year(
            year
        )

        # -----------------------------
        # Check dimensions
        # -----------------------------

        assert (
            ds.sizes["valid_time"]
            == expected_hours
        )

        assert (
            ds.sizes["study_id"]
            == 512
        )

        # -----------------------------
        # Check required variables
        # -----------------------------

        assert "u10" in ds.data_vars
        assert "v10" in ds.data_vars

        # -----------------------------
        # Check spatial alignment
        # -----------------------------

        assert np.array_equal(
            ds["study_id"].values,
            study_grid["study_id"].to_numpy()
        )

        assert np.array_equal(
            ds["grid_id"].values,
            study_grid["grid_id"].to_numpy()
        )

        assert np.allclose(
            ds["latitude"].values,
            study_grid["latitude"].to_numpy()
        )

        assert np.allclose(
            ds["longitude"].values,
            study_grid["longitude"].to_numpy()
        )

        # -----------------------------
        # Check time coverage
        # -----------------------------

        times = pd.DatetimeIndex(
            pd.to_datetime(
                ds["valid_time"].values
            )
        )

        expected_start = pd.Timestamp(
            f"{year}-01-01 00:00:00"
        )

        expected_end = pd.Timestamp(
            f"{year}-12-31 23:00:00"
        )

        assert times[0] == expected_start
        assert times[-1] == expected_end

        assert not times.has_duplicates

        time_differences = (
            pd.Series(times)
            .diff()
            .dropna()
        )

        assert (
            time_differences
            == pd.Timedelta(hours=1)
        ).all()

        # -----------------------------
        # Check missing values
        # -----------------------------

        u10_missing = int(
            ds["u10"]
            .isnull()
            .sum()
            .values
        )

        v10_missing = int(
            ds["v10"]
            .isnull()
            .sum()
            .values
        )

        assert u10_missing == 0
        assert v10_missing == 0

        # -----------------------------
        # Record result
        # -----------------------------

        final_check_results.append({
            "year": year,
            "start_time": times[0],
            "end_time": times[-1],
            "hours": len(times),
            "study_cells": ds.sizes["study_id"],
            "u10_missing": u10_missing,
            "v10_missing": v10_missing,
            "status": "PASSED"
        })


final_check = pd.DataFrame(
    final_check_results
)

print("\nFinal integrity check:")
display(final_check)

print(
    "\nFinal integrity check: PASSED"
)

Number of processed files: 16
Checking 2005 ...
Checking 2006 ...
Checking 2007 ...
Checking 2008 ...
Checking 2009 ...
Checking 2010 ...
Checking 2011 ...
Checking 2012 ...
Checking 2013 ...
Checking 2014 ...
Checking 2015 ...
Checking 2016 ...
Checking 2017 ...
Checking 2018 ...
Checking 2019 ...
Checking 2020 ...

Final integrity check:


,year,start_time,end_time,hours,study_cells,u10_missing,v10_missing,status
0,2005,2005-01-01,2005-12-31 23:00:00,8760,512,0,0,PASSED
1,2006,2006-01-01,2006-12-31 23:00:00,8760,512,0,0,PASSED
2,2007,2007-01-01,2007-12-31 23:00:00,8760,512,0,0,PASSED
3,2008,2008-01-01,2008-12-31 23:00:00,8784,512,0,0,PASSED
4,2009,2009-01-01,2009-12-31 23:00:00,8760,512,0,0,PASSED
5,2010,2010-01-01,2010-12-31 23:00:00,8760,512,0,0,PASSED
6,2011,2011-01-01,2011-12-31 23:00:00,8760,512,0,0,PASSED
7,2012,2012-01-01,2012-12-31 23:00:00,8784,512,0,0,PASSED
8,2013,2013-01-01,2013-12-31 23:00:00,8760,512,0,0,PASSED
9,2014,2014-01-01,2014-12-31 23:00:00,8760,512,0,0,PASSED



Final integrity check: PASSED


In [16]:
# Display the final preprocessing output structure

print("Preprocessing completed successfully.\n")

print("Processed data directory:")
print(YEARLY_DIR.resolve())

print("\nProcessed annual files:")
for file in processed_files:
    print(f"- {file.name}")

print("\nPreprocessing summary file:")
print(PREPROCESSING_SUMMARY_PATH.resolve())

print("\nTotal annual files:", len(processed_files))
print("Study-grid cells:", len(study_grid))
print(
    "Persian Gulf cells:",
    (study_grid["basin"] == "Persian Gulf").sum()
)
print(
    "Gulf of Oman cells:",
    (study_grid["basin"] == "Gulf of Oman").sum()
)

print(
    "\n02_preprocessing.ipynb: COMPLETED"
)

Preprocessing completed successfully.

Processed data directory:
/home/mike/Desktop/wind_paper/data/processed/yearly

Processed annual files:
- 2005.nc
- 2006.nc
- 2007.nc
- 2008.nc
- 2009.nc
- 2010.nc
- 2011.nc
- 2012.nc
- 2013.nc
- 2014.nc
- 2015.nc
- 2016.nc
- 2017.nc
- 2018.nc
- 2019.nc
- 2020.nc

Preprocessing summary file:
/home/mike/Desktop/wind_paper/outputs/tables/preprocessing_summary.csv

Total annual files: 16
Study-grid cells: 512
Persian Gulf cells: 357
Gulf of Oman cells: 155

02_preprocessing.ipynb: COMPLETED
